In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/polar-test/subtask2/test/tel.csv
/kaggle/input/polar-test/subtask2/test/mya.csv
/kaggle/input/polar-test/subtask2/test/amh.csv
/kaggle/input/polar-test/subtask2/test/eng.csv
/kaggle/input/polar-test/subtask2/test/ben.csv
/kaggle/input/polar-test/subtask2/test/hau.csv
/kaggle/input/polar-test/subtask2/test/khm.csv
/kaggle/input/polar-test/subtask2/test/pan.csv
/kaggle/input/polar-test/subtask2/test/pol.csv
/kaggle/input/polar-test/subtask2/test/arb.csv
/kaggle/input/polar-test/subtask2/test/hin.csv
/kaggle/input/polar-test/subtask2/test/nep.csv
/kaggle/input/polar-test/subtask2/test/ita.csv
/kaggle/input/polar-test/subtask2/test/zho.csv
/kaggle/input/polar-test/subtask2/test/urd.csv
/kaggle/input/polar-test/subtask2/test/rus.csv
/kaggle/input/polar-test/subtask2/test/swa.csv
/kaggle/input/polar-test/subtask2/test/ori.csv
/kaggle/input/polar-test/subtask2/test/deu.csv
/kaggle/input/polar-test/subtask2/test/spa.csv
/kaggle/input/polar-test/subtask2/test/tur.csv
/kaggle/input

In [2]:
import os
import glob
import random
import math
import numpy as np
import pandas as pd
from typing import Any, Dict, List, Optional

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

from sklearn.metrics import f1_score, hamming_loss

# ==================== Kaggle/GPU sanity ====================
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ==================== Configuration ====================
LABEL_KEYS = [
    "stereotype",
    "vilification",
    "dehumanization",
    "extreme_language",
    "lack_of_empathy",
    "invalidation"
]

SEED = 42
MAX_LEN = 256

FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

THRESH_GRID = np.linspace(0.05, 0.95, 37)

LANG_BALANCE_POWER = 0.5
LABEL_AWARE_WEIGHT = 0.4
WEIGHT_CLIP_PCT = (5, 95)

USE_LANG_LABEL_LOSS_SCALING = True
LOSS_SCALE_POWER = 0.50
LOSS_SCALE_CLIP = (0.8, 1.6)

USE_RDROP = True
RDROP_ALPHA = 0.6
RDROP_PROB = 0.5

USE_LORA_IF_AVAILABLE = True

# ==================== Utilities ====================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def normalize_columns(df):
    df.columns = (
        df.columns.str.replace("\n", " ")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

seed_everything(SEED)

# ==================== Load and Prepare Data (Kaggle paths) ====================
# Your requirement: dataset name "polar_test" and training uses subtask_3/training
KAGGLE_TRAIN_PATH = "/kaggle/input/polar-test/subtask3/train"

possible_paths = [
    KAGGLE_TRAIN_PATH,
    "/kaggle/input/polar-test/subtask3/train",     # fallback, just in case
    "/kaggle/input/polar_test/training",            # fallback
    "./subtask_3/training",
    "./training",
]

base_path = None
for p in possible_paths:
    if os.path.exists(p) and len(glob.glob(os.path.join(p, "*.csv"))) >= 2:
        base_path = p
        break

if base_path is None:
    # show what Kaggle mounted to help you debug quickly
    print("Contents of /kaggle/input:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "N/A")
    raise ValueError(f"Training data not found. Expected CSVs under: {KAGGLE_TRAIN_PATH}")

print("Using training path:", base_path)

dfs = []
for f in glob.glob(os.path.join(base_path, "*.csv")):
    lang = os.path.basename(f).split(".")[0]
    d = pd.read_csv(f)
    d["lang"] = lang
    dfs.append(d)

df_all = normalize_columns(pd.concat(dfs, ignore_index=True))

# Ensure required columns exist
if "text" not in df_all.columns:
    raise ValueError("Expected a 'text' column in training CSVs after normalization.")

for k in LABEL_KEYS:
    if k not in df_all.columns:
        df_all[k] = 0
    df_all[k] = df_all[k].fillna(0).astype(int)

all_langs = sorted(df_all["lang"].unique())
lang2id = {l: i for i, l in enumerate(all_langs)}

print("\nLanguages:", all_langs)

# ==================== Train/Val Split ====================
def split_by_language(df, val_ratio=0.15):
    tr, va = [], []
    rng = np.random.RandomState(SEED)
    for _, sub in df.groupby("lang"):
        idx = np.arange(len(sub))
        rng.shuffle(idx)
        n_val = max(1, int(len(sub) * val_ratio))
        va.append(sub.iloc[idx[:n_val]])
        tr.append(sub.iloc[idx[n_val:]])
    return pd.concat(tr).reset_index(drop=True), pd.concat(va).reset_index(drop=True)

train, val = split_by_language(df_all)

# ==================== lang_label_mask ====================
def build_lang_label_mask(df):
    mask = np.zeros((len(all_langs), len(LABEL_KEYS)), dtype=np.float32)
    for lang in all_langs:
        sub = df[df.lang == lang]
        if len(sub) == 0:
            continue
        mask[lang2id[lang]] = (sub[LABEL_KEYS].sum(0).values > 0).astype(float)
    return torch.tensor(mask)

lang_label_mask = build_lang_label_mask(train)

# ==================== Dataset Classes ====================
class PolarizationDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.labels = df[LABEL_KEYS].values.astype(np.float32)
        self.lang_ids = df.lang.map(lang2id).values.astype(np.int64)

        texts = df.text.astype(str).tolist()
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding=False,
            max_length=max_length,
        )

    def __len__(self):
        return len(self.lang_ids)

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        item["lang_id"] = torch.tensor(self.lang_ids[i])
        return item


class PredictionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length=256):
        self.texts = df["text"].astype(str).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in enc.items()}

# ==================== Custom Data Collator ====================
class CustomDataCollator(DataCollatorWithPadding):
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        batch = super().__call__(features)

        if "labels" in batch:
            batch["labels"] = batch["labels"].float()
            if "lang_id" in features[0]:
                batch["lang_id"] = torch.stack([f["lang_id"] for f in features])

        if "sample_weight" in features[0]:
            batch["sample_weight"] = torch.stack([f["sample_weight"] for f in features]).float()

        return batch

# ==================== Global + Per-language pos_weight ====================
def compute_pos_weight(df, eps=1.0):
    y = df[LABEL_KEYS].values
    pos = y.sum(0)
    neg = len(y) - pos
    return torch.tensor((neg + eps) / (pos + eps), dtype=torch.float32)

def compute_lang_label_pos_weight(df, eps=1.0):
    mat = np.zeros((len(all_langs), len(LABEL_KEYS)))
    for lang in all_langs:
        sub = df[df.lang == lang]
        y = sub[LABEL_KEYS].values
        pos = y.sum(0)
        neg = len(y) - pos
        mat[lang2id[lang]] = (neg + eps) / (pos + eps)
    return torch.tensor(mat, dtype=torch.float32)

global_pos_weight = compute_pos_weight(train)
lang_label_pos_weight = compute_lang_label_pos_weight(train)

print("\nGlobal pos_weight (N_neg/N_pos) per label:")
for k, w in zip(LABEL_KEYS, global_pos_weight.tolist()):
    print(f"  {k:18s} {w:.3f}")

if USE_LANG_LABEL_LOSS_SCALING:
    ratio = (lang_label_pos_weight / global_pos_weight.unsqueeze(0)).clamp(min=1e-6)
    loss_scale = ratio.pow(LOSS_SCALE_POWER).clamp(min=LOSS_SCALE_CLIP[0], max=LOSS_SCALE_CLIP[1])
else:
    loss_scale = torch.ones_like(lang_label_pos_weight)

# ==================== Language–label-aware sampler weights ====================
def build_example_weights(df: pd.DataFrame) -> np.ndarray:
    lang_counts = df["lang"].value_counts().to_dict()
    inv_lang = {l: (1.0 / c) ** LANG_BALANCE_POWER for l, c in lang_counts.items()}

    y = df[LABEL_KEYS].values.astype(np.float32)
    lang_ids = df["lang"].map(lang2id).values.astype(int)
    pw_lang = lang_label_pos_weight.detach().cpu().numpy()

    pos_mask = y > 0.5
    pos_weight_sum = (pos_mask * pw_lang[lang_ids]).sum(axis=1)
    pos_count = pos_mask.sum(axis=1).astype(np.float32)

    pos_mean = np.zeros(len(df), dtype=np.float32)
    nz = pos_count > 0
    pos_mean[nz] = pos_weight_sum[nz] / pos_count[nz]

    label_factor = 1.0 + (LABEL_AWARE_WEIGHT * pos_mean)
    lang_factor = np.array([inv_lang[l] for l in df["lang"].tolist()], dtype=np.float32)

    w = (lang_factor * label_factor).astype(np.float32)
    lo, hi = np.percentile(w, WEIGHT_CLIP_PCT[0]), np.percentile(w, WEIGHT_CLIP_PCT[1])
    w = np.clip(w, lo, hi)
    w = w / (w.mean() + 1e-8)

    print("\nSampler diagnostics:")
    print(f"  weights: min={w.min():.3f}, mean={w.mean():.3f}, max={w.max():.3f}")
    print("  language counts (train):", dict(sorted(lang_counts.items(), key=lambda x: x[0])))

    return w

train_example_weights = build_example_weights(train)

def asymmetric_loss_with_logits(logits, targets, pos_weight):
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight.to(logits.device)
    )
    pt = torch.sigmoid(logits) * targets + (1 - torch.sigmoid(logits)) * (1 - targets)
    return ((1 - pt) ** FOCAL_GAMMA) * bce

def bernoulli_kl(p, q, eps=1e-6):
    p = p.clamp(eps, 1 - eps)
    q = q.clamp(eps, 1 - eps)
    return p * (p / q).log() + (1 - p) * ((1 - p) / (1 - q)).log()

# ==================== Metrics (monitoring at 0.5) ====================
def compute_metrics_multilabel(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid_np(logits)
    preds = (probs >= 0.5).astype(int)
    labels_int = labels.astype(int)

    per_label_f1 = f1_score(labels_int, preds, average=None, zero_division=0)
    metrics = {
        "f1_macro": f1_score(labels_int, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels_int, preds, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(labels_int, preds),
    }
    for k, s in zip(LABEL_KEYS, per_label_f1):
        metrics[f"f1_{k}"] = float(s)
    return metrics

# ==================== Threshold tuning ====================
def tune_thresholds_global(val_logits, val_true):
    probs = sigmoid_np(val_logits)
    th = np.zeros(probs.shape[1], dtype=float)
    for j in range(probs.shape[1]):
        best_f1, best_t = -1.0, 0.5
        for t in THRESH_GRID:
            pred = (probs[:, j] >= t).astype(int)
            f1 = f1_score(val_true[:, j], pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)
        th[j] = best_t
    return th

def tune_thresholds_per_language(val_df: pd.DataFrame, val_logits: np.ndarray):
    global_th = tune_thresholds_global(val_logits, val_df[LABEL_KEYS].values.astype(int))
    probs = sigmoid_np(val_logits)
    out = {}

    for lang in sorted(val_df["lang"].unique().tolist()):
        idx = np.where(val_df["lang"].values == lang)[0]
        y = val_df.iloc[idx][LABEL_KEYS].values.astype(int)
        p = probs[idx]

        th = np.array(global_th, copy=True)
        for j in range(len(LABEL_KEYS)):
            if y[:, j].sum() == 0 or y[:, j].sum() == len(y):
                continue
            best_f1, best_t = -1.0, th[j]
            for t in THRESH_GRID:
                pred = (p[:, j] >= t).astype(int)
                f1 = f1_score(y[:, j], pred, zero_division=0)
                if f1 > best_f1:
                    best_f1 = f1
                    best_t = float(t)
            th[j] = best_t

        out[lang] = th

    print("\nSample of per-language thresholds (first 5 langs):")
    for lang in list(out.keys())[:5]:
        print(f"  {lang}: {np.round(out[lang], 2)}")

    return out, global_th

def apply_lang_thresholds(df: pd.DataFrame, logits: np.ndarray, th_by_lang: dict, global_th: np.ndarray):
    probs = sigmoid_np(logits)
    preds = np.zeros_like(probs, dtype=int)
    for i, lang in enumerate(df["lang"].tolist()):
        th = th_by_lang.get(lang, global_th)
        preds[i] = (probs[i] >= th).astype(int)
    return preds, probs

# ==================== Calibration: Per-language Temperature Scaling ====================
@torch.no_grad()
def _bce_eval_loss(logits_t: torch.Tensor, labels_t: torch.Tensor, pos_weight: torch.Tensor) -> float:
    loss = nn.functional.binary_cross_entropy_with_logits(
        logits_t, labels_t, reduction="mean", pos_weight=pos_weight.to(logits_t.device)
    )
    return float(loss.item())

def fit_temperature_per_language(
    val_df: pd.DataFrame,
    val_logits: np.ndarray,
    label_keys: List[str],
    lang2id: Dict[str, int],
    pos_weight: torch.Tensor,
    device: Optional[str] = None,
    max_iter: int = 200,
    lr: float = 0.05,
    t_min: float = 0.5,
    t_max: float = 5.0,
    min_samples: int = 200,
) -> Dict[str, float]:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    logits_all = torch.tensor(val_logits, dtype=torch.float32, device=device)
    y_all = torch.tensor(val_df[label_keys].values.astype(np.float32), device=device)

    langs = sorted(val_df["lang"].unique().tolist())

    global_log_t = torch.zeros(1, device=device, requires_grad=True)
    opt = torch.optim.LBFGS([global_log_t], lr=lr, max_iter=max_iter)

    def closure_global():
        opt.zero_grad()
        T = torch.exp(global_log_t).clamp(min=t_min, max=t_max)
        loss = nn.functional.binary_cross_entropy_with_logits(
            logits_all / T, y_all, reduction="mean", pos_weight=pos_weight.to(device)
        )
        loss.backward()
        return loss

    opt.step(closure_global)
    T_global = float(torch.exp(global_log_t).clamp(min=t_min, max=t_max).item())

    temps = {lang: T_global for lang in langs}

    for lang in langs:
        idx = np.where(val_df["lang"].values == lang)[0]
        if len(idx) < min_samples:
            temps[lang] = T_global
            continue

        logits_lang = logits_all[idx]
        y_lang = y_all[idx]

        log_t = torch.tensor([math.log(T_global)], device=device, requires_grad=True)
        opt_lang = torch.optim.LBFGS([log_t], lr=lr, max_iter=max_iter)

        def closure_lang():
            opt_lang.zero_grad()
            T = torch.exp(log_t).clamp(min=t_min, max=t_max)
            loss = nn.functional.binary_cross_entropy_with_logits(
                logits_lang / T, y_lang, reduction="mean", pos_weight=pos_weight.to(device)
            )
            loss.backward()
            return loss

        opt_lang.step(closure_lang)
        temps[lang] = float(torch.exp(log_t).clamp(min=t_min, max=t_max).item())

    print("\nTemperature scaling (T per language):")
    for lang in list(temps.keys())[:8]:
        print(f"  {lang}: {temps[lang]:.3f}")

    return temps

def apply_temperature_per_language(
    df: pd.DataFrame,
    logits: np.ndarray,
    temps_by_lang: Dict[str, float],
    default_T: float = 1.0
) -> np.ndarray:
    out = logits.astype(np.float32).copy()
    langs = df["lang"].tolist()
    for i, lang in enumerate(langs):
        T = temps_by_lang.get(lang, default_T)
        if T <= 0:
            T = default_T
        out[i] = out[i] / float(T)
    return out

def evaluate_macro_f1_from_logits_with_lang_thresholds(
    df: pd.DataFrame,
    logits: np.ndarray,
    th_by_lang: Dict[str, np.ndarray],
    global_th: np.ndarray,
    label_keys: List[str]
) -> float:
    preds, _ = apply_lang_thresholds(df, logits, th_by_lang, global_th)
    y_true = df[label_keys].values.astype(int)
    return float(f1_score(y_true, preds, average="macro", zero_division=0))

# ==================== Custom Trainer ====================
class FocalLossLangAwareTrainer(Trainer):
    def __init__(self, pos_weight=None, loss_scale=None, alpha=0.25, gamma=2.0, sample_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
        self.loss_scale = loss_scale
        self.alpha = alpha
        self.gamma = gamma
        self.sample_weights = sample_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels", None)
        lang_id = inputs.pop("lang_id", None)
    
        outputs = model(**inputs)
        logits = outputs.logits
    
        # If labels aren't provided, let HF handle it (shouldn't happen in train/eval)
        if labels is None:
            loss = None
            return (loss, outputs) if return_outputs else loss
    
        # Base focal/asymmetric loss elementwise: [B, L]
        loss_elem = asymmetric_loss_with_logits(logits, labels, global_pos_weight)
    
        # If lang_id exists, apply mask+scale; else fallback to mean loss
        if lang_id is not None:
            mask = lang_label_mask.to(logits.device)[lang_id]
            scale = self.loss_scale.to(logits.device)[lang_id]
            loss = (loss_elem * mask * scale).mean()
        else:
            loss = loss_elem.mean()
    
        # R-Drop only if lang_id exists (optional), otherwise skip safely
        if USE_RDROP and (lang_id is not None) and (torch.rand(1).item() < RDROP_PROB):
            outputs2 = model(**inputs)
            logits2 = outputs2.logits
            mask = lang_label_mask.to(logits.device)[lang_id]
            kl = bernoulli_kl(torch.sigmoid(logits), torch.sigmoid(logits2))
            kl = (kl * mask).mean()
            loss = loss + RDROP_ALPHA * kl
    
        return (loss, outputs) if return_outputs else loss


    def get_train_dataloader(self):
        if self.sample_weights is None:
            return super().get_train_dataloader()
    
        n = len(self.train_dataset)
        w = np.asarray(self.sample_weights)
    
        if len(w) != n:
            raise ValueError(
                f"sample_weights length ({len(w)}) != train_dataset length ({n}). "
                "Recompute weights on the SAME 'train' used for train_dataset."
            )
    
        sampler = WeightedRandomSampler(
            weights=torch.tensor(w, dtype=torch.float),
            num_samples=n,          # ✅ full epoch coverage
            replacement=True
        )
    
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=True
        )


    # def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
    #     inputs.pop("lang_id", None)
    #     return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys)


# ==================== Model and Tokenizer ====================
MODEL_NAME = "Sami92/XLM-R-Large-Polarization-Classifier"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_KEYS),
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True
)

# try:
#     model.gradient_checkpointing_enable()
#     print("\n✓ Enabled gradient checkpointing")
# except Exception as e:
#     print("\n(i) Gradient checkpointing not enabled:", str(e))

if USE_LORA_IF_AVAILABLE:
    try:
        from peft import LoraConfig, get_peft_model, TaskType
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=["query", "key", "value"]
        )
        model = get_peft_model(model, lora_config)
        print("\n✓ Using LoRA adapters (PEFT)")
        try:
            model.print_trainable_parameters()
        except Exception:
            pass
    except Exception as e:
        print("\n(i) PEFT/LoRA not available; continuing without LoRA.")
        print("    Reason:", str(e))

# ==================== Datasets ====================
train_dataset = PolarizationDataset(train, tokenizer, max_length=MAX_LEN)
val_dataset = PolarizationDataset(val, tokenizer, max_length=MAX_LEN)

data_collator = CustomDataCollator(tokenizer)

# ==================== Training Arguments (Kaggle-safe) ====================
training_args = TrainingArguments(
    output_dir="./checkpoints_xlmr_large_langaware",
    num_train_epochs=6,
    learning_rate=4e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    warmup_ratio=0.10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    seed=SEED,
    save_total_limit=2,

    # Kaggle-safe optimizer (fused often breaks depending on image)
    optim="adamw_torch",

    gradient_accumulation_steps=2,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,  # more stable on Kaggle than 4
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,  # keep lang_id
)

# ==================== Trainer ====================
trainer = FocalLossLangAwareTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_multilabel,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    pos_weight=global_pos_weight,
    loss_scale=loss_scale,
    alpha=FOCAL_ALPHA,
    gamma=FOCAL_GAMMA,
    sample_weights=train_example_weights
)

# ==================== Train ====================
print("\nDEBUG:")
print("len(train):", len(train))
print("len(train_dataset):", len(train_dataset))
print("len(train_example_weights):", len(train_example_weights))
print("expected steps/epoch:",
      math.ceil(len(train_dataset) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)))

print("\n" + "="*60)
print("Starting training...")
print("="*60 + "\n")
trainer.train()

# ==================== Evaluate (0.5 monitoring) ====================
eval_results = trainer.evaluate()
print("\n" + "="*60)
print("Validation Results (threshold=0.5; monitoring only)")
print("="*60)
print(f"Macro F1: {eval_results['eval_f1_macro']:.4f}")
print(f"Micro F1: {eval_results['eval_f1_micro']:.4f}")
print(f"Hamming Loss: {eval_results['eval_hamming_loss']:.4f}")

# ==================== Tune thresholds per language ====================
val_pred = trainer.predict(val_dataset)
val_logits = val_pred.predictions

temps_by_lang = fit_temperature_per_language(
    val_df=val,
    val_logits=val_logits,
    label_keys=LABEL_KEYS,
    lang2id=lang2id,
    pos_weight=global_pos_weight,
    min_samples=200
)

val_logits_cal = apply_temperature_per_language(val, val_logits, temps_by_lang, default_T=1.0)
th_by_lang, global_th = tune_thresholds_per_language(val, val_logits_cal)

val_preds_lang, _ = apply_lang_thresholds(val, val_logits_cal, th_by_lang, global_th)
val_true = val[LABEL_KEYS].values.astype(int)

val_macro_lang = f1_score(val_true, val_preds_lang, average="macro", zero_division=0)
val_micro_lang = f1_score(val_true, val_preds_lang, average="micro", zero_division=0)
val_hl_lang = hamming_loss(val_true, val_preds_lang)

macro_before = evaluate_macro_f1_from_logits_with_lang_thresholds(val, val_logits, th_by_lang, global_th, LABEL_KEYS)
macro_after  = evaluate_macro_f1_from_logits_with_lang_thresholds(val, val_logits_cal, th_by_lang, global_th, LABEL_KEYS)
print(f"\nVal Macro-F1 (uncalibrated logits): {macro_before:.4f}")
print(f"Val Macro-F1 (temp-scaled logits):   {macro_after:.4f}")

print("\n" + "="*60)
print("Validation Results (language-specific thresholds)")
print("="*60)
print(f"Macro F1: {val_macro_lang:.4f}")
print(f"Micro F1: {val_micro_lang:.4f}")
print(f"Hamming Loss: {val_hl_lang:.4f}")


2026-01-30 18:59:26.697604: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769799566.937572      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769799567.009992      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769799567.568410      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769799567.568451      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769799567.568454      24 computation_placer.cc:177] computation placer alr

CUDA available: True
GPU: Tesla T4
Using training path: /kaggle/input/polar-test/subtask3/train

Languages: ['amh', 'arb', 'ben', 'deu', 'eng', 'fas', 'hau', 'hin', 'khm', 'nep', 'ori', 'pan', 'spa', 'swa', 'tel', 'tur', 'urd', 'zho']

Global pos_weight (N_neg/N_pos) per label:
  stereotype         1.974
  vilification       2.210
  dehumanization     7.641
  extreme_language   3.555
  lack_of_empathy    4.296
  invalidation       5.079

Sampler diagnostics:
  weights: min=0.419, mean=1.000, max=2.817
  language counts (train): {'amh': 2833, 'arb': 2873, 'ben': 2834, 'deu': 2703, 'eng': 2739, 'fas': 2801, 'hau': 3104, 'hin': 2333, 'khm': 5644, 'nep': 1705, 'ori': 2013, 'pan': 1445, 'spa': 2810, 'swa': 5943, 'tel': 2012, 'tur': 2010, 'urd': 3029, 'zho': 3638}


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...


✓ Using LoRA adapters (PEFT)
trainable params: 3,415,046 || all params: 563,311,628 || trainable%: 0.6062

DEBUG:
len(train): 52469
len(train_dataset): 52469
len(train_example_weights): 52469
expected steps/epoch: 3280

Starting training...



Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Hamming Loss,F1 Stereotype,F1 Vilification,F1 Dehumanization,F1 Extreme Language,F1 Lack Of Empathy,F1 Invalidation,Runtime,Samples Per Second,Steps Per Second
1,0.302300,0.233904,0.479979,0.487863,0.339477,0.537418,0.627664,0.327345,0.523364,0.452655,0.411427,300.171400,30.816000,1.929000
2,0.283700,0.220361,0.505705,0.516903,0.347604,0.637101,0.645754,0.329075,0.503493,0.461248,0.457561,299.190600,30.917000,1.935000
3,0.275000,0.206469,0.532678,0.549959,0.294919,0.662867,0.656836,0.377997,0.524211,0.493701,0.480453,298.578400,30.980000,1.939000
4,0.270200,0.210077,0.523829,0.537777,0.330577,0.663892,0.664718,0.381093,0.534877,0.455945,0.442450,300.485600,30.784000,1.927000
5,0.264700,0.208090,0.525230,0.540108,0.326324,0.663453,0.664553,0.369299,0.528663,0.470824,0.454591,301.335800,30.697000,1.921000



Validation Results (threshold=0.5; monitoring only)
Macro F1: 0.5327
Micro F1: 0.5500
Hamming Loss: 0.2949


/usr/local/lib/python3.12/dist-packages/torch/optim/lbfgs.py:457: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  loss = float(closure())



Temperature scaling (T per language):
  amh: 0.500
  arb: 0.500
  ben: 0.500
  deu: 0.637
  eng: 0.500
  fas: 0.500
  hau: 0.500
  hin: 0.500

Sample of per-language thresholds (first 5 langs):
  amh: [0.48 0.55 0.65 0.52 0.35 0.42]
  arb: [0.57 0.78 0.62 0.75 0.57 0.57]
  ben: [0.22 0.5  0.8  0.45 0.3  0.18]
  deu: [0.6  0.6  0.6  0.65 0.5  0.48]
  eng: [0.37 0.57 0.57 0.6  0.52 0.45]

Val Macro-F1 (uncalibrated logits): 0.4818
Val Macro-F1 (temp-scaled logits):   0.5524

Validation Results (language-specific thresholds)
Macro F1: 0.5524
Micro F1: 0.5751
Hamming Loss: 0.2604


In [3]:
import shutil

# ==================== Load Dev Data (Kaggle) ====================
DEV_KAGGLE_PATH = "/kaggle/input/polar-test/subtask3/dev"

dev_possible_paths = [
    DEV_KAGGLE_PATH,                         # ✅ Kaggle
    "/kaggle/input/polar-test/subtask3/dev",  # same, explicit
    "./subtask3/dev",                         # local fallback
    "./dev",                                  # local fallback
]

dev_base_path = None
for p in dev_possible_paths:
    if os.path.exists(p) and glob.glob(os.path.join(p, "*.csv")):
        dev_base_path = p
        break

if dev_base_path is None:
    # quick debug
    if os.path.exists("/kaggle/input"):
        print("Mounted datasets:", os.listdir("/kaggle/input"))
    raise ValueError(f"Dev data folder not found. Tried: {dev_possible_paths}")

print("Using dev path:", dev_base_path)

all_dev_files = glob.glob(os.path.join(dev_base_path, "*.csv"))
dev_df_list = []
for f in all_dev_files:
    lang_code = os.path.basename(f).split('.')[0]
    temp_df = pd.read_csv(f)
    temp_df['lang'] = lang_code
    dev_df_list.append(temp_df)

dev_all = normalize_columns(pd.concat(dev_df_list, ignore_index=True))
if "text" not in dev_all.columns or "id" not in dev_all.columns:
    raise ValueError("Dev CSVs must include 'id' and 'text' columns.")

missing_langs = sorted(set(dev_all["lang"].unique()) - set(all_langs))
if missing_langs:
    raise ValueError(f"Dev has languages not seen in train mapping: {missing_langs}")

print(f"\nDevelopment set size: {len(dev_all)}")
print(f"Languages: {sorted(dev_all['lang'].unique())}")


Using dev path: /kaggle/input/polar-test/subtask3/dev

Development set size: 3091
Languages: ['amh', 'arb', 'ben', 'deu', 'eng', 'fas', 'hau', 'hin', 'khm', 'nep', 'ori', 'pan', 'spa', 'swa', 'tel', 'tur', 'urd', 'zho']


In [4]:
# ==================== Save Predictions ====================
pred_dataset = PredictionDataset(dev_all, tokenizer, max_length=MAX_LEN)
dev_pred = trainer.predict(pred_dataset)
dev_logits = dev_pred.predictions

dev_logits_cal = apply_temperature_per_language(dev_all, dev_logits, temps_by_lang, default_T=1.0)
dev_labels_bin, _ = apply_lang_thresholds(dev_all, dev_logits_cal, th_by_lang, global_th)

pred_df = dev_all.copy()
for j, label in enumerate(LABEL_KEYS):
    pred_df[label] = dev_labels_bin[:, j].astype(int)

output_dir = "/kaggle/working/subtask_3"   # ✅ Kaggle output location
os.makedirs(output_dir, exist_ok=True)

for lang in sorted(pred_df["lang"].unique()):
    out = pred_df[pred_df["lang"] == lang][["id"] + LABEL_KEYS]
    out.to_csv(os.path.join(output_dir, f"pred_{lang}.csv"), index=False)
    print(f"Saved {len(out)} predictions for {lang}")

zip_path = shutil.make_archive("/kaggle/working/subtask_3", "zip", output_dir)
print(f"\n✓ Created: {zip_path}")
print("In Kaggle: open the right-side 'Output' panel and download subtask_3.zip")


Saved 166 predictions for amh
Saved 169 predictions for arb
Saved 166 predictions for ben
Saved 159 predictions for deu
Saved 160 predictions for eng
Saved 164 predictions for fas
Saved 182 predictions for hau
Saved 137 predictions for hin
Saved 332 predictions for khm
Saved 100 predictions for nep
Saved 118 predictions for ori
Saved 100 predictions for pan
Saved 165 predictions for spa
Saved 349 predictions for swa
Saved 118 predictions for tel
Saved 115 predictions for tur
Saved 177 predictions for urd
Saved 214 predictions for zho

✓ Created: /kaggle/working/subtask_3.zip
In Kaggle: open the right-side 'Output' panel and download subtask_3.zip


In [5]:
import os
import json
import numpy as np
import torch

# ==================== Kaggle save directory ====================
BASE_DIR = "/kaggle/working/testing"
os.makedirs(BASE_DIR, exist_ok=True)

# Model-specific subfolder
save_path = os.path.join(BASE_DIR, "polarization_modelB_Sami")
os.makedirs(save_path, exist_ok=True)

# ==================== Save model ====================
# If using LoRA (PEFT), this correctly saves adapters
model.save_pretrained(save_path)

# Save tokenizer
tokenizer.save_pretrained(save_path)

# ==================== Save metadata ====================
serializable_thresholds = {
    lang: th.tolist() if isinstance(th, np.ndarray) else th
    for lang, th in th_by_lang.items()
}

metadata = {
    "label_keys": LABEL_KEYS,
    "global_thresholds": global_th.tolist(),
    "per_language_thresholds": serializable_thresholds,
    "per_language_temperatures": temps_by_lang,
    "model_name": MODEL_NAME
}

with open(os.path.join(save_path, "post_processing_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print(f"✓ Model and metadata saved to {save_path}")
print("✓ Download from Kaggle → Output panel")


✓ Model and metadata saved to /kaggle/working/testing/polarization_modelB_Sami
✓ Download from Kaggle → Output panel


In [6]:
import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/polarization_modelB_Sami",
    "zip",
    save_path
)

print(f"✓ Zipped model at: {zip_path}")
print("→ Download polarization_modelB_Sami.zip from Kaggle Outputs")


✓ Zipped model at: /kaggle/working/polarization_modelB_Sami.zip
→ Download polarization_modelB_Sami.zip from Kaggle Outputs
